### Template notebook for running pipeline

In [13]:
%load_ext autoreload
%autoreload 2
    
from create_featurized_object import (load_data, 
    merge_and_qc_data, 
    create_features_and_metadata,
    create_anndata_object,
    export_anndata_object
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Create featurized object

In [2]:
# Setting up filenames and paths
library_path = '/seqlrg/LRS_POC/SeqCenter_QUO1014840/b.00025'
genome_path = '/srv/shared/genomes/'
genome = 'H_elongata'

In [3]:
# Load data
barcodes, mapped_inserts, inserts_only = load_data(library_path, genome_path, genome)

print('Data loaded.')

Data loaded.


In [14]:
# Merge and QC data
merge = merge_and_qc_data(barcodes, inserts_only)

In [15]:
# Create features and metadata
features_full, features_partial, bc_meta_avg, features_meta, empty_barcodes = create_features_and_metadata(
    merge,
    barcodes,
    mapped_inserts)

/home/jupyter-jhkliu42/jonathan/long-read-library-featurization/bin/create_featurized_object.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mapped_inserts['overlap_ohc'] = [1 if x == 1 else 0 for x in mapped_inserts.overlap_frac]
/home/jupyter-jhkliu42/jonathan/long-read-library-featurization/bin/create_featurized_object.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/home/jupyter-jhkliu42/jonathan/long-read-library-featurization/bin/create_featurized_object.py:136: SettingWithCopyWarning: 
A

In [16]:
# Create AnnData object
obj = create_anndata_object(features_full, features_partial, bc_meta_avg, features_meta, empty_barcodes)

In [17]:
obj.obs

,bc_length,insert_chr,insert_start,insert_end,insert_length,insert_sense
bc_sequence,,,,,,
AAAAAAAAAAAAAGCAGATTCCCATTCGTGTCGCG,35,NC_014532.2,717291,720761,3470,+
AAAAAAAAAAAACAGCAGATAATGATATGTTAAGGTGTG,39,NC_014532.2,3373230,3377089,3859,+
AAAAAAAAAAAAGAACAGCAGAGTTCGCTTGTAGACTTC,39,NC_014532.2,916665,919191,2526,+
AAAAAAAAAAAAGACAATGGTCATCTGTTTG,31,NC_014532.2,2939085,2940835,1750,+
AAAAAAAAAAACAGTGTTTTCTGTTCCGTAC,31,NC_014532.2,1898902,1903368,4466,-
...,...,...,...,...,...,...
TTTTTTTATCACATAAGCAGACTGTCTTCTTATCTAGGATG,41,NC_014532.2,2277976,2282651,4675,+
TTTTTTTCTTCACACAGAGATTTTTGTCTCTCTC,34,NC_014532.2,3757238,3759913,2675,+
TTTTTTTGAATTGATAGCAGAAATTGGTGTGTTAGTTTC,39,NC_014532.2,2832875,2835832,2957,-


In [21]:
# Export AnnData object
out_path = library_path
filename = 'featurized_object.h5ad'

export_anndata_object(obj, out_path, filename)

print('Object exported.')

Object exported.


### Integrate object with short-read fitness data
We are likely going to change our barcode error correction scheme with ground truth detection of barcode libraries in the future, so I don't think it's worth pipelining this part yet. See this notebook for reference: https://github.com/Pioneer-Research-Labs/analysis-notebooks/blob/main/LRS_POC_analysis/barcode_correction_investigation.ipynb